# Compare Scenarios

Runs `run_compare_scenarios.py` logic inline — configure in the **Configuration** cell, then run all cells.

In [5]:
import sys
from pathlib import Path

# Locate repo root and put src/ on the path
here = Path(".").resolve()
for _p in [here, *here.parents]:
    if (_p / "pyproject.toml").exists():
        REPO = _p
        break
else:
    raise RuntimeError("Could not find repo root (no pyproject.toml)")

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

print(f"Repo root: {REPO}")

Repo root: /Users/annavisman/stack/TUDelft/thesis/msc-thesis


In [6]:
import importlib, sys

# Force-reload in case the module was already imported
for mod in list(sys.modules):
    if mod.startswith("thesis"):
        del sys.modules[mod]

import matplotlib
matplotlib.use("inline")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Import helpers from the script
import importlib.util, types

_script = REPO / "src/thesis/scripts/run_compare_scenarios.py"
_spec = importlib.util.spec_from_file_location("run_compare_scenarios", _script)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

# Pull names into local scope for convenience
_run_compare       = _mod._run_compare
_load_compare_result = _mod._load_compare_result
_build_long_table  = _mod._build_long_table
_build_delta_table = _mod._build_delta_table
_format_text_table = _mod._format_text_table
plot_auc_comparison   = _mod.plot_auc_comparison
plot_metrics_breakdown = _mod.plot_metrics_breakdown
plot_feature_counts   = _mod.plot_feature_counts
plot_fp_comparison    = _mod.plot_fp_comparison
EXPERIMENTS_DIR       = _mod.EXPERIMENTS_DIR

print("Imports OK")

Imports OK


## Configuration

Edit the variables below, then run the remaining cells.

In [18]:
# ── Edit these ──────────────────────────────────────────────
SCENARIOS     = ["fox"]
FORCE         = True   # True  → re-run even if results already exist
NO_RUN        = False   # True  → only load existing results, skip running
FILTER_CONFIG = REPO / "src/thesis/configs/mining_filters_discriminative.yaml"
# FILTER_CONFIG = None
# ────────────────────────────────────────────────────────────

## Run comparisons

In [19]:
all_results = []

for scenario in SCENARIOS:
    existing = _load_compare_result(scenario)

    if NO_RUN:
        if existing is not None:
            print(f"[{scenario}] Loaded existing results ({existing['source']})")
            all_results.append(existing)
        else:
            print(f"[{scenario}] No existing results — skipping (NO_RUN=True).")
        continue

    if existing is not None and not FORCE:
        print(f"[{scenario}] Skipping — result already exists ({existing['source']}). Set FORCE=True to re-run.")
        all_results.append(existing)
        continue

    r = _run_compare(scenario, filter_config=FILTER_CONFIG)
    all_results.append(r)

print(f"\nTotal results: {len(all_results)}")


 Running compare: fox

--- Phase 1/2: baseline ---

[Baseline] Scenario: 'fox'
[1/6] Converting alerts to JSON...
  [skip] alerts.json already exists at /Users/annavisman/stack/TUDelft/thesis/msc-thesis/artifacts/processed-data/fox/alerts.json
[2/7] Processing alert batch...
  [skip] Alert cache already populated at /Users/annavisman/stack/TUDelft/thesis/msc-thesis/artifacts/cache/fox/alerts
[3/7] Checking feature manifest...
  [skip] Feature manifest already exists at /Users/annavisman/stack/TUDelft/thesis/msc-thesis/artifacts/features/fox/manifest.json
[4/7] Building transactions from cache...
  [skip] Loading transactions from existing /Users/annavisman/stack/TUDelft/thesis/msc-thesis/artifacts/cache/fox/transactions/transactions_raw.json
  Loaded 10310 transactions from cache.
[5/7] Encoding transactions (schema='base')...
  [skip] Loading encoded transactions from existing /Users/annavisman/stack/TUDelft/thesis/msc-thesis/artifacts/cache/fox/transactions/transactions_base.parquet

## Build tables

In [ ]:
long_df  = _build_long_table(all_results)
delta_df = _build_delta_table(all_results)

subdir_name = "_".join(SCENARIOS)
out_dir = EXPERIMENTS_DIR / "plots" / subdir_name
out_dir.mkdir(parents=True, exist_ok=True)

csv_path  = out_dir / "compare_table.csv"
delta_csv = out_dir / "compare_delta_table.csv"
txt_path  = out_dir / "compare_table.txt"

long_df.to_csv(csv_path, index=False)
delta_df.to_csv(delta_csv, index=False)

text_table = _format_text_table(delta_df)
txt_path.write_text(text_table, encoding="utf-8")

print(text_table)

In [ ]:
display(delta_df)

## Plots

In [ ]:
%matplotlib inline
plot_auc_comparison(delta_df, out_dir)
plt.show()

In [ ]:
plot_metrics_breakdown(delta_df, out_dir)
plt.show()

In [ ]:
plot_feature_counts(delta_df, out_dir)
plt.show()

In [ ]:
plot_fp_comparison(delta_df, out_dir)
plt.show()